In [1]:
import os
import argparse
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
'''
🤗 Datasets is a library for easily accessing and sharing AI datasets for Audio, Computer Vision, and Natural Language Processing (NLP) tasks.

Load a dataset in a single line of code, and use our powerful data processing and streaming methods to quickly get your dataset ready for 
training in a deep learning model. Backed by the Apache Arrow format, process large datasets with zero-copy reads without any memory 
constraints for optimal speed and efficiency. 

We also feature a deep integration with the Hugging Face Hub, allowing you to easily load and share a dataset with the wider machine learning community.
'''
'''
“Zero-copy reads”
This means data is read without being copied from one memory buffer to another.
Normally, data reading looks like:
Disk → kernel buffer → user buffer → application


With zero-copy, one or more of those copies are eliminated:
Disk → kernel → application (direct access)
'''
from datasets import Dataset
from peft import LoraConfig, get_peft_model

try:
    from vllm import LLM, SamplingParams
except Exception:
    LLM = None
    SamplingParams = None

INFO 12-23 18:51:20 [__init__.py:216] Automatically detected platform cpu.


In [2]:
def make_train_dataset(tokenizer):
    # Small toy dataset of "instruction -> response" pairs.
    samples = [
        {"text": "Q: Summarize the following text: Python is a popular language.\nA: Python is a widely used programming language known for readability."},
        {"text": "Q: Translate to Spanish: Hello\nA: Hola"},
        {"text": "Q: What is 2 + 2?\nA: 4"},
    ]
    ds = Dataset.from_list(samples)

    def tokenize_fn(examples):
        toks = tokenizer(examples["text"], truncation=True, max_length=128)
        return toks

    ds = ds.map(tokenize_fn, remove_columns=["text"])
    ds.set_format(type="torch")
    return ds

In [3]:
def generate_with_vllm(model_dir, prompt, temperature=0.7):
    if LLM is None:
        raise RuntimeError("vllm is not installed or failed to import.")

    llm = LLM(model=model_dir, \
              max_model_len=8192,
              max_num_batched_tokens=8192,
              dtype="float16"
        )
    sampling_params = SamplingParams(temperature=temperature)

    outputs = llm.generate(prompt, sampling_params=sampling_params)
    for out in outputs:
        # out.text is incremental; join full_text if needed. Keep simple:
        print("=== vllm generation ===")
        print(out.outputs[0].text)
    return outputs

In [4]:
response = generate_with_vllm(
    model_dir = "/Users/tmittra/models/Qwen2-1.5B-Instruct",
    prompt = "Tell me a joke"
)

INFO 12-23 18:51:21 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 8192, 'max_num_batched_tokens': 8192, 'disable_log_stats': True, 'model': '/Users/tmittra/models/Qwen2-1.5B-Instruct'}


INFO 12-23 18:51:21 [model.py:547] Resolved architecture: Qwen2ForCausalLM
WARNING 12-23 18:51:21 [model.py:1733] Casting torch.bfloat16 to torch.float16.
INFO 12-23 18:51:21 [model.py:1510] Using max model len 8192
WARNING 12-23 18:51:21 [cpu.py:117] Environment variable VLLM_CPU_KVCACHE_SPACE (GiB) for CPU backend is not set, using 4 by default.
INFO 12-23 18:51:21 [arg_utils.py:1166] Chunked prefill is not supported for ARM and POWER and S390X CPUs; disabling it for V1 backend.
INFO 12-23 18:51:27 [__init__.py:216] Automatically detected platform cpu.
(EngineCore_DP0 pid=67597) INFO 12-23 18:51:28 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=67597) INFO 12-23 18:51:28 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/Users/tmittra/models/Qwen2-1.5B-Instruct', speculative_config=None, tokenizer='/Users/tmittra/models/Qwen2-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_

[W1223 18:51:29.618661000 ProcessGroupGloo.cpp:545] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:05<00:00,  5.38s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:05<00:00,  5.38s/it]
(EngineCore_DP0 pid=67597) 


(EngineCore_DP0 pid=67597) INFO 12-23 18:51:35 [default_loader.py:267] Loading weights took 5.39 seconds
(EngineCore_DP0 pid=67597) INFO 12-23 18:51:35 [kv_cache_utils.py:1087] GPU KV cache size: 149,792 tokens
(EngineCore_DP0 pid=67597) INFO 12-23 18:51:35 [kv_cache_utils.py:1091] Maximum concurrency for 8,192 tokens per request: 18.29x
(EngineCore_DP0 pid=67597) INFO 12-23 18:51:35 [cpu_model_runner.py:117] Warming up model for the compilation...
(EngineCore_DP0 pid=67597) WARNING 12-23 18:51:35 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=67597) INFO 12-23 18:51:43 [cpu_model_runner.py:121] Warming up done.
(EngineCore_DP0 pid=67597) INFO 12-23 18:51:43 [core.py:210] init engine (profile, create kv cache, warmup model) took 8.08 seconds
(EngineCore_DP0 pid=67597) WARNING 12-23 18:51:43 [cpu.py:117] Environment variable VLLM_CPU_KVCACHE_SPACE (GiB) for CPU backend is not set, using 4 by default.
INFO 12-2

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

=== vllm generation ===
 about math.
Sure, here's a joke about math:
Why did the math


In [6]:
response = generate_with_vllm(
    model_dir = "/Users/tmittra/models/Qwen2-1.5B-Instruct",
    prompt = '''System: You are a helpful assistant.
User: What is 2 + 2?
    ''',
    temperature=0.7
)

INFO 12-23 18:52:12 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 8192, 'max_num_batched_tokens': 8192, 'disable_log_stats': True, 'model': '/Users/tmittra/models/Qwen2-1.5B-Instruct'}
INFO 12-23 18:52:12 [model.py:547] Resolved architecture: Qwen2ForCausalLM
WARNING 12-23 18:52:12 [model.py:1733] Casting torch.bfloat16 to torch.float16.
INFO 12-23 18:52:12 [model.py:1510] Using max model len 8192
INFO 12-23 18:52:12 [arg_utils.py:1166] Chunked prefill is not supported for ARM and POWER and S390X CPUs; disabling it for V1 backend.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 12-23 18:52:17 [__init__.py:216] Automatically detected platform cpu.
(EngineCore_DP0 pid=67813) INFO 12-23 18:52:17 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=67813) INFO 12-23 18:52:17 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/Users/tmittra/models/Qwen2-1.5B-Instruct', speculative_config=None, tokenizer='/Users/tmittra/models/Qwen2-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cpu, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_config=Observa

[W1223 18:52:18.914399000 ProcessGroupGloo.cpp:545] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.75s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.75s/it]
(EngineCore_DP0 pid=67813) 


(EngineCore_DP0 pid=67813) INFO 12-23 18:52:21 [default_loader.py:267] Loading weights took 2.75 seconds
(EngineCore_DP0 pid=67813) INFO 12-23 18:52:21 [kv_cache_utils.py:1087] GPU KV cache size: 149,792 tokens
(EngineCore_DP0 pid=67813) INFO 12-23 18:52:21 [kv_cache_utils.py:1091] Maximum concurrency for 8,192 tokens per request: 18.29x
(EngineCore_DP0 pid=67813) INFO 12-23 18:52:22 [cpu_model_runner.py:117] Warming up model for the compilation...
(EngineCore_DP0 pid=67813) WARNING 12-23 18:52:22 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=67813) INFO 12-23 18:52:30 [cpu_model_runner.py:121] Warming up done.
(EngineCore_DP0 pid=67813) INFO 12-23 18:52:30 [core.py:210] init engine (profile, create kv cache, warmup model) took 8.20 seconds
(EngineCore_DP0 pid=67813) WARNING 12-23 18:52:30 [cpu.py:117] Environment variable VLLM_CPU_KVCACHE_SPACE (GiB) for CPU backend is not set, using 4 by default.
INFO 12-2

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

=== vllm generation ===
!!!!!!!!!!!!!!!!


In [7]:
response

[RequestOutput(request_id=0, prompt='System: You are a helpful assistant.\nUser: What is 2 + 2?\n    ', prompt_token_ids=[2320, 25, 1446, 525, 264, 10950, 17847, 624, 1474, 25, 3555, 374, 220, 17, 488, 220, 17, 5267, 257], encoder_prompt=None, encoder_prompt_token_ids=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text='!!!!!!!!!!!!!!!!', token_ids=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], cumulative_logprob=None, logprobs=None, finish_reason=length, stop_reason=None)], finished=True, metrics=None, lora_request=None, num_cached_tokens=0, multi_modal_placeholders={})]

In [8]:
response = generate_with_vllm(
    model_dir = "/Users/tmittra/models/Qwen2-1.5B-Instruct",
    prompt = '''Tell me a joke.
    ''',
    temperature=0.7
)

INFO 12-23 18:52:40 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 8192, 'max_num_batched_tokens': 8192, 'disable_log_stats': True, 'model': '/Users/tmittra/models/Qwen2-1.5B-Instruct'}
INFO 12-23 18:52:40 [model.py:547] Resolved architecture: Qwen2ForCausalLM
WARNING 12-23 18:52:40 [model.py:1733] Casting torch.bfloat16 to torch.float16.
INFO 12-23 18:52:40 [model.py:1510] Using max model len 8192
INFO 12-23 18:52:40 [arg_utils.py:1166] Chunked prefill is not supported for ARM and POWER and S390X CPUs; disabling it for V1 backend.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 12-23 18:52:44 [__init__.py:216] Automatically detected platform cpu.
(EngineCore_DP0 pid=67927) INFO 12-23 18:52:45 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=67927) INFO 12-23 18:52:45 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/Users/tmittra/models/Qwen2-1.5B-Instruct', speculative_config=None, tokenizer='/Users/tmittra/models/Qwen2-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cpu, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_config=Observa

[W1223 18:52:46.638903000 ProcessGroupGloo.cpp:545] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=67927) INFO 12-23 18:52:46 [parallel_state.py:1208] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=67927) INFO 12-23 18:52:46 [cpu_model_runner.py:106] Starting to load model /Users/tmittra/models/Qwen2-1.5B-Instruct...
(EngineCore_DP0 pid=67927) INFO 12-23 18:52:46 [cpu.py:104] Using Torch SDPA backend.


Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:03<00:00,  3.11s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:03<00:00,  3.12s/it]
(EngineCore_DP0 pid=67927) 


(EngineCore_DP0 pid=67927) INFO 12-23 18:52:49 [default_loader.py:267] Loading weights took 3.12 seconds
(EngineCore_DP0 pid=67927) INFO 12-23 18:52:49 [kv_cache_utils.py:1087] GPU KV cache size: 149,792 tokens
(EngineCore_DP0 pid=67927) INFO 12-23 18:52:49 [kv_cache_utils.py:1091] Maximum concurrency for 8,192 tokens per request: 18.29x
(EngineCore_DP0 pid=67927) INFO 12-23 18:52:50 [cpu_model_runner.py:117] Warming up model for the compilation...
(EngineCore_DP0 pid=67927) WARNING 12-23 18:52:50 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=67927) INFO 12-23 18:52:57 [cpu_model_runner.py:121] Warming up done.
(EngineCore_DP0 pid=67927) INFO 12-23 18:52:57 [core.py:210] init engine (profile, create kv cache, warmup model) took 7.84 seconds
(EngineCore_DP0 pid=67927) WARNING 12-23 18:52:58 [cpu.py:117] Environment variable VLLM_CPU_KVCACHE_SPACE (GiB) for CPU backend is not set, using 4 by default.
INFO 12-2

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

=== vllm generation ===
!!!!!!!!!!!!!!!!
